# ReDrafter Extended Training — Qwen2.5-7B on A100
**Extended run targeting 1.5-1.8x speedup on Apple Silicon**

Trains a larger RNN draft head (Apple arxiv:2403.09919)
for token-level speculative decoding.

**Runtime:** GPU → A100 (Colab Pro)
**Expected training time:** ~8-12 hours (50,000 steps)
**Expected speedup after deployment:** 1.5-1.8x on Apple Silicon

Changes vs quick run:
- `DRAFTER_HIDDEN`: 512 → 1024 (bigger head, higher acceptance rate)
- `DRAFTER_LAYERS`: 2 → 4 (deeper GRU)
- `NUM_DRAFT_TOKENS`: 4 → 6 (more speculative parallelism)
- `MAX_SEQ_LEN`: 512 → 1024 (better real-world coverage)
- `BATCH_SIZE`: 8 → 16
- `MAX_SAMPLES`: 50K → full ShareGPT (94K)
- `MAX_STEPS`: 3,000 → 50,000
- Checkpoints every 2,500 steps (Google Drive backup)

⏰ **Tip:** Enable Colab Pro+ to keep GPU allocated for 24h

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate bitsandbytes sentencepiece
print('✅ Dependencies installed')

In [ ]:
# Mount Google Drive for checkpoint backup (recommended for long runs)
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_OUTPUT = '/content/drive/MyDrive/redrafter_extended'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'✅ Drive mounted. Checkpoints will save to: {DRIVE_OUTPUT}')

In [ ]:
import os, gc, math, time, logging, shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import numpy as np

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
log = logging.getLogger()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

In [ ]:
# ── Extended Config ──────────────────────────────────────────────────────
BASE_MODEL        = 'Qwen/Qwen2.5-7B-Instruct'
DRAFTER_HIDDEN    = 1024   # 2x bigger (was 512)
DRAFTER_LAYERS    = 4      # 2x deeper (was 2)
NUM_DRAFT_TOKENS  = 6      # +2 tokens (was 4)
MAX_SEQ_LEN       = 1024   # 2x longer context (was 512)
MAX_SAMPLES       = 94145  # Full ShareGPT dataset
BATCH_SIZE        = 16     # 2x batch (was 8)
GRAD_ACCUM        = 2      # Effective batch = 32 (same as before)
LEARNING_RATE     = 1e-4   # Lower LR for longer run
WARMUP_STEPS      = 1000
MAX_STEPS         = 50_000
SAVE_EVERY        = 2500   # Save every ~30 min
LOG_EVERY         = 100
OUTPUT_DIR        = '/content/drafter_output'
DRIVE_OUTPUT      = '/content/drive/MyDrive/redrafter_extended'  # set in cell above
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Resume from checkpoint if available
RESUME_FROM = None  # e.g. '/content/drive/MyDrive/redrafter_extended/redrafter_step25000.pt'

print('✅ Extended config set')
print(f'   Draft head: {DRAFTER_HIDDEN}d x {DRAFTER_LAYERS}L x {NUM_DRAFT_TOKENS} tokens')
print(f'   Steps: {MAX_STEPS:,} | Batch: {BATCH_SIZE} | Seq: {MAX_SEQ_LEN}')

In [ ]:
# ── Load base model (frozen) ─────────────────────────────────────────────
print(f'Loading tokenizer: {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
vocab_size = len(tokenizer)
print(f'Vocab size: {vocab_size}')

print(f'Loading base model (bfloat16, CUDA)...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)
base_model.eval()
for p in base_model.parameters():
    p.requires_grad_(False)

hidden_size = base_model.config.hidden_size
print(f'✅ Base model loaded. Hidden size: {hidden_size}')

In [ ]:
# ── ReDrafter Head (Extended) ────────────────────────────────────────────
class ReDrafterHead(nn.Module):
    def __init__(self, vocab_size, hidden_size, drafter_hidden=1024, num_layers=4, num_draft_tokens=6):
        super().__init__()
        self.num_draft_tokens = num_draft_tokens
        self.input_proj  = nn.Linear(hidden_size, drafter_hidden)
        self.gru = nn.GRU(drafter_hidden, drafter_hidden, num_layers=num_layers, batch_first=True)
        self.token_embed = nn.Embedding(vocab_size, drafter_hidden)
        self.output_proj = nn.Linear(drafter_hidden, vocab_size, bias=False)
        # Layer norm for stability in deeper network
        self.layer_norm  = nn.LayerNorm(drafter_hidden)
        nn.init.xavier_uniform_(self.input_proj.weight)
        nn.init.zeros_(self.input_proj.bias)
        nn.init.xavier_uniform_(self.output_proj.weight)

    def forward(self, hidden_state, target_ids=None):
        B = hidden_state.size(0)
        proj = self.layer_norm(self.input_proj(hidden_state))
        h0 = proj.unsqueeze(0).expand(self.gru.num_layers, -1, -1).contiguous()
        x = torch.zeros(B, h0.shape[-1], device=hidden_state.device, dtype=hidden_state.dtype)
        h, logits_list = h0, []
        for step in range(self.num_draft_tokens):
            out, h = self.gru(x.unsqueeze(1), h)
            logits = self.output_proj(out.squeeze(1))
            logits_list.append(logits)
            x = self.token_embed(target_ids[:, step] if target_ids is not None else logits.argmax(-1))
        return torch.stack(logits_list, dim=1)  # [B, K, V]

draft_head = ReDrafterHead(
    vocab_size, hidden_size, DRAFTER_HIDDEN, DRAFTER_LAYERS, NUM_DRAFT_TOKENS
).to(device).to(torch.bfloat16)

n_params = sum(p.numel() for p in draft_head.parameters())
print(f'✅ Draft head: {n_params:,} params ({n_params/1e6:.1f}M)')

# Resume from checkpoint if specified
start_step = 0
if RESUME_FROM and os.path.exists(RESUME_FROM):
    draft_head.load_state_dict(torch.load(RESUME_FROM, map_location=device))
    start_step = int(RESUME_FROM.split('step')[-1].split('.')[0])
    print(f'✅ Resumed from step {start_step}: {RESUME_FROM}')

In [ ]:
# ── Full ShareGPT Dataset ─────────────────────────────────────────────────
class ShareGPTDataset(Dataset):
    def __init__(self, tokenizer, max_seq=1024, max_samples=94145):
        print(f'Loading ShareGPT (max {max_samples} samples, seq_len={max_seq})...')
        ds = load_dataset(
            'anon8231489123/ShareGPT_Vicuna_unfiltered',
            data_files='ShareGPT_V3_unfiltered_cleaned_split.json',
            split='train'
        )
        self.samples = []
        for i, row in enumerate(ds.select(range(min(max_samples, len(ds))))):
            text = ' '.join(t.get('value','') for t in row.get('conversations',[]))
            if len(text) < 200: continue
            ids = tokenizer(text, truncation=True, max_length=max_seq+1, return_tensors='pt').input_ids.squeeze(0)
            if len(ids) >= NUM_DRAFT_TOKENS + 4: self.samples.append(ids)
            if i % 10000 == 0 and i > 0: print(f'  Tokenized {i}/{max_samples}...')
        print(f'✅ Dataset ready: {len(self.samples)} samples')

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def collate(batch):
    ml = max(x.size(0) for x in batch)
    p = torch.zeros(len(batch), ml, dtype=torch.long)
    for i,x in enumerate(batch): p[i,:x.size(0)] = x
    return p

dataset = ShareGPTDataset(tokenizer, MAX_SEQ_LEN, MAX_SAMPLES)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, drop_last=True, num_workers=2)

In [ ]:
# ── Extended Training Loop ────────────────────────────────────────────────
try:
    from bitsandbytes.optim import AdamW8bit
    optimizer = AdamW8bit(draft_head.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    print('Using 8-bit AdamW (memory efficient)')
except Exception:
    from torch.optim import AdamW
    optimizer = AdamW(draft_head.parameters(), lr=LEARNING_RATE, weight_decay=0.01, betas=(0.9,0.95))
    print('Using standard AdamW')

def get_lr(step):
    if step < WARMUP_STEPS: return LEARNING_RATE * step / max(WARMUP_STEPS, 1)
    p = (step - WARMUP_STEPS) / max(MAX_STEPS - WARMUP_STEPS, 1)
    return LEARNING_RATE * 0.5 * (1 + math.cos(math.pi * p))

step = start_step
optimizer.zero_grad()
t0 = time.time()
t_last_log = t0
new_lr = LEARNING_RATE
best_loss = float('inf')

print(f'Starting extended training from step {step}/{MAX_STEPS}...')
print(f'Estimated completion: {MAX_STEPS * 0.9 / 3600:.1f}h (rough estimate)')

while step < MAX_STEPS:
    for batch in loader:
        if step >= MAX_STEPS: break
        batch = batch.to(device)
        sl = batch.size(1)
        if sl < NUM_DRAFT_TOKENS + 4: continue
        start_pos = torch.randint(1, max(2, sl - NUM_DRAFT_TOKENS - 1), (1,)).item()
        ctx = batch[:, :start_pos]
        tgt = batch[:, start_pos:start_pos + NUM_DRAFT_TOKENS]

        with torch.no_grad():
            out = base_model(input_ids=ctx, output_hidden_states=True, use_cache=False)
            h = out.hidden_states[-1][:, -1, :]

        logits = draft_head(h, target_ids=tgt)
        loss = sum(F.cross_entropy(logits[:,k,:].float(), tgt[:,k], ignore_index=0)
                   for k in range(NUM_DRAFT_TOKENS)) / NUM_DRAFT_TOKENS
        (loss / GRAD_ACCUM).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            new_lr = get_lr(step)
            for pg in optimizer.param_groups: pg['lr'] = new_lr
            optimizer.step()
            optimizer.zero_grad()

        step += 1
        loss_val = loss.item()
        if loss_val < best_loss: best_loss = loss_val

        if step % LOG_EVERY == 0:
            elapsed = time.time() - t0
            eta_h = (MAX_STEPS - step) * (elapsed / step) / 3600
            steps_per_sec = step / elapsed
            print(f'step={step}/{MAX_STEPS} | loss={loss_val:.4f} | best={best_loss:.4f} | '
                  f'lr={new_lr:.2e} | {elapsed/3600:.2f}h elapsed | ETA {eta_h:.1f}h | '
                  f'{steps_per_sec:.1f} steps/s')

        if step % SAVE_EVERY == 0:
            # Save locally
            local_path = f'{OUTPUT_DIR}/redrafter_step{step}.pt'
            torch.save(draft_head.state_dict(), local_path)
            # Backup to Drive
            drive_path = f'{DRIVE_OUTPUT}/redrafter_step{step}.pt'
            shutil.copy(local_path, drive_path)
            print(f'  💾 Checkpoint saved: step {step} → Drive ({os.path.getsize(drive_path)/1e6:.1f}MB)')

        gc.collect()
        if step % 500 == 0:
            torch.cuda.empty_cache()

# Final save
final_local = f'{OUTPUT_DIR}/redrafter_final.pt'
final_drive = f'{DRIVE_OUTPUT}/redrafter_final.pt'
torch.save(draft_head.state_dict(), final_local)
shutil.copy(final_local, final_drive)
total_time = time.time() - t0
print(f'\n✅ Extended training complete!')
print(f'   Total time: {total_time/3600:.2f}h')
print(f'   Best loss: {best_loss:.4f}')
print(f'   Final weights: {final_drive}')

In [ ]:
# ── Download final weights ───────────────────────────────────────────────
# Weights are already in Google Drive — this also downloads to local
import shutil
shutil.make_archive('/content/redrafter_extended_final', 'gztar', OUTPUT_DIR)
from google.colab import files
files.download('/content/redrafter_extended_final.tar.gz')
print('✅ Weights downloaded!')
print('Weights also saved to Google Drive:', DRIVE_OUTPUT)